# Word guesser: build the word table with gemma4:12b

Word list: the THINGS concept list (Hebart et al., 2019, PLoS ONE) plus a few extra words and
39 wildcard words (nothing, the internet, love...). 129 questions.

**Before running:** right-hand settings panel -> Accelerator **GPU T4 x2**, Internet **on**.
Run the cells top to bottom, one at a time.

1. Cells 1-3 install Ollama and the helper libraries (a few minutes).
2. The `%%writefile` cells create the project files.
3. The **test run** scores 40 words and prints the seconds per word.
4. The last cell checks the table.
5. For the **full run**: change `LIMIT=40` to `LIMIT=0` in the run cell, then use
   **Save Version -> Save & Run All (Commit)** so it keeps running even if you close the tab.
   Progress is saved after every word, so a stopped run resumes where it left off.
6. When it finishes, download `words.json` from the notebook's Output section.

Everything installed here disappears when the session ends.


In [ ]:
!nvidia-smi

In [ ]:
# Install Ollama and the Python libraries
!apt-get install -y -qq lshw pciutils zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q -U ollama wordfreq


In [ ]:
%%writefile questions.py
"""The question list shared by the scoring notebook and the game.

Each question is about the TYPICAL example of the thing. Keep each one
under 33 characters so it fits on the 16x2 LCD.
"""
QUESTIONS = {
    "alive": "Is it alive?",
    "animal": "Is it an animal?",
    "plant": "Is it a plant or part of one?",
    "humanbody": "Is it part of the human body?",
    "manmade": "Was it made by people?",
    "food": "Is it food or a drink?",
    "fruitveg": "Is it a fruit or vegetable?",
    "drink": "Is it a drink?",
    "meat": "Does it contain meat or fish?",
    "sweet": "Does it taste sweet?",
    "savory": "Does it taste salty or savory?",
    "baked": "Is it baked, fried or grilled?",
    "raw": "Can you eat it raw?",
    "pocket": "Fits in a pocket?",
    "hand": "Can you hold it in one hand?",
    "bigperson": "Is it bigger than a person?",
    "bigcar": "Is it bigger than a car?",
    "heavy": "Is it too heavy to carry alone?",
    "round": "Is it mostly round?",
    "longthin": "Is it long and thin?",
    "flat": "Is it mostly flat?",
    "hollow": "Can it hold things inside?",
    "handle": "Does it have a handle?",
    "pair": "Does it come in pairs?",
    "metal": "Is it mostly metal?",
    "wood": "Is it mostly wood?",
    "plastic": "Is it mostly plastic?",
    "glass": "Is it mostly glass or ceramic?",
    "fabric": "Is it mostly fabric or leather?",
    "paper": "Is it mostly paper or cardboard?",
    "stone": "Is it mostly stone or rock?",
    "soft": "Is it soft to touch?",
    "shiny": "Is it shiny?",
    "seethrough": "Can you see through it?",
    "breakable": "Does it break easily?",
    "folds": "Can it be folded?",
    "opens": "Does it open and close?",
    "movingparts": "Does it have moving parts?",
    "buttons": "Does it have buttons or keys?",
    "screen": "Does it have a screen?",
    "electric": "Does it use electricity?",
    "wheels": "Does it have wheels?",
    "engine": "Does it have an engine?",
    "light": "Does it give off light?",
    "sound": "Does it make sound?",
    "loud": "Is it loud?",
    "hot": "Is it usually hot or warm?",
    "cold": "Is it usually cold?",
    "wet": "Is it wet or a liquid?",
    "smell": "Does it have a strong smell?",
    "sharp": "Does it have a sharp part?",
    "danger": "Can it seriously hurt you?",
    "moves": "Can it move by itself?",
    "flies": "Can it fly?",
    "swims": "Can it swim?",
    "legs": "Does it have legs?",
    "fourlegs": "Does it have four legs?",
    "furfeather": "Does it have fur or feathers?",
    "scales": "Does it have scales or a shell?",
    "tail": "Does it have a tail?",
    "eyes": "Does it have eyes?",
    "pet": "Is it a common pet?",
    "farm": "Is it found on a farm?",
    "leaves": "Does it have leaves?",
    "flowerfruit": "Does it have flowers or fruit?",
    "grows": "Does it grow from the ground?",
    "inhome": "Is one usually in a home?",
    "kitchen": "Is it usually in a kitchen?",
    "bathroom": "Is it usually in a bathroom?",
    "bedroom": "Is it usually in a bedroom?",
    "livingroom": "Is it usually in a living room?",
    "garage": "Is it kept in a garage or shed?",
    "school": "Is it found in a classroom?",
    "office": "Is it found in an office?",
    "outdoors": "Is it mainly used outdoors?",
    "wild": "Is it found in the wild?",
    "water": "Is it found in or on water?",
    "sky": "Is it found in the sky?",
    "space": "Is it found in outer space?",
    "city": "Is it found in cities or towns?",
    "store": "Can you buy it in a store?",
    "medical": "Is it used in medicine?",
    "tool": "Is it a tool?",
    "toy": "Is it a toy or game?",
    "sport": "Is it used in sports?",
    "music": "Is it used to make music?",
    "wear": "Do people wear it?",
    "wearhead": "Is it worn on the head?",
    "wearfeet": "Is it worn on the feet?",
    "vehicle": "Is it a vehicle?",
    "cook": "Is it used for cooking?",
    "eatwith": "Used for eating or drinking?",
    "clean": "Is it used for cleaning?",
    "write": "Is it used to write or draw?",
    "read": "Do you read it?",
    "cut": "Is it used to cut things?",
    "storage": "Is it used to store things?",
    "protect": "Does it protect you?",
    "sit": "Can you sit or lie on it?",
    "decor": "Is it used for decoration?",
    "holiday": "Is it used at holidays?",
    "building": "Is it a building or structure?",
    "inside": "Can you go inside it?",
    "money": "Is it related to money?",
    "kids": "Is it mainly for children?",
    "expensive": "Does it cost over $100?",
    "weekly": "Do most people use it weekly?",
    "disposable": "Is it thrown away after use?",
    "tiny": "Is it smaller than a coin?",
    "bigdog": "Is it bigger than a dog?",
    "green": "Is it often green?",
    "redpink": "Is it often red or pink?",
    "yelloworange": "Is it often yellow or orange?",
    "brown": "Is it often brown?",
    "white": "Is it often white?",
    "blackgray": "Is it often black or gray?",
    "bluepurple": "Is it often blue or purple?",
    "juicy": "Is it juicy?",
    "crunchy": "Is it crunchy?",
    "pit": "Does it have a pit or seeds?",
    "hotplace": "Is it from a hot climate?",
    "coldplace": "Is it found in cold places?",
    "eatsmeat": "Does it eat other animals?",
    "touch": "Can you touch it?",
    "idea": "Is it an idea or a feeling?",
    "place": "Is it a place?",
    "activity": "Is it an activity or event?",
    "online": "Does it exist mostly online?",
    "person": "Is it a person or a job?",
}


In [ ]:
%%writefile word_list.txt
aardvark
abacus
accordion
acorn
air conditioner
air mattress
air pump
airbag
airboat
aircraft carrier
airplane
album
alligator
almond
aloe
alpaca
altar
aluminum foil
amber
ambulance
amplifier
anchor
ankle
anklet
ant
anteater
antelope
antenna
anvil
appetizer
apple
apple tree
applesauce
apron
aquarium
arch
arm
armor
arrow
artichoke
arugula
ashtray
asparagus
avocado
awning
axe
baby
backdrop
backgammon
backpack
backscratcher
bacon
badge
badger
bag
bagel
bagpipe
baklava
balance beam
ball
balloon
ballot box
bamboo
banana
banana peel
banana split
bandage
bandanna
banjo
bank
banner
barbed wire
barbell
barcode
bark
barnacle
barrel
barrette
baseball
baseball bat
baseball glove
basil
basket
basketball
basketball hoop
bassinet
bassoon
bat
bathmat
bathrobe
bathtub
baton
batter
battery
bazooka
beachball
bead
beaker
bean
beanbag
beanie
bear
beard
beaver
bed
bedpan
bedpost
bee
beehive
beer
beet
beetle
bell
bell pepper
belt
belt buckle
bench
berry
bib
bike
bikini
bin
binder
binoculars
bird
birdbath
birdcage
birdhouse
birthday cake
biscuit
bison
blackberry
blanket
blazer
blender
blimp
blind
blinder
blindfold
block
blouse
blower
blowfish
blowgun
blowtorch
blueberry
boa
boar
board
board game
boat
bobsled
bok choy
bolo tie
bologna
bolt
bomb
bone
bongo
bonnet
bonsai
book
bookmark
bookshelf
boomerang
boot
bottle
bottle opener
boulder
bouquet
bow
bowl
bowler hat
bowling ball
bowtie
box
boxer shorts
boxing gloves
boy
bra
brace
bracelet
bracket
braid
brake
branch
brass knuckles
bread
breadbox
breadstick
breakfast
breathalyzer
brick
briefcase
broccoli
brooch
broom
brownie
brush
brussels sprouts
bubble
bubble wrap
bucket
buckle
buffet
bug
buggy
bull
bulldozer
bullet
bulletin board
bulletproof vest
bumper
bun
bungee
bunkbed
buoy
burner
burrito
bus
bush
butter
butterfly
button
cabbage
cabinet
cable
cactus
cage
cake
cake mix
calamari
calculator
calf
calzone
camcorder
camel
camera lens
camera
camper
can
can opener
candelabra
candle
candlestick
candy
candy bar
candy cane
cane
canister
cannon
cannonball
canoe
cantaloupe
canvas
cap
cape
car
car door
car seat
caramel
card
cardboard
cardigan
cardinal
carousel
carriage
carrot
cart
cash machine
cash register
cashew
casserole
cassette
cat
catapult
caterpillar
catfish
cauliflower
caviar
cd player
celery
cello
cellphone
cement mixer
centerpiece
centrifuge
cereal
chain
chainsaw
chair
chalice
chalk
chalkboard
champagne
chandelier
chaps
charcoal
charger
chariot
checkbook
checkers
cheese
cheeseburger
cheesecake
cheetah
cherry
chess piece
chessboard
chest
chick
chicken wire
chicken
chickpea
chihuahua
chili
chime
chimney
chin
chinaware
chinchilla
chip
chipmunk
chips
chisel
chive
chocolate
chopsticks
christmas card
christmas tree
chute
cigar
cigarette
cigarette butt
cigarette holder
cilantro
cinnamon
clam
clarinet
clasp
clay
cleat
cleaver
clipboard
clipper
cloak
clock
closet
clothes
clothesline
clothespin
cloud
clove
clover
coal
coaster
coat
coat rack
cobra
cockatoo
cockroach
cocktail
coconut
cocoon
coffee
coffee bean
coffee filter
coffee pot
coffee table
coffeemaker
coffin
coil
coin
coleslaw
collar
column
comb
combination lock
comic book
compass
compost
computer
computer screen
confetti
contact lens
container
cooker
cookie
cookie cutter
cookie sheet
cooler
coop
copier
coral
cord
cordon bleu
cork
corkboard
corkscrew
corn
cornbread
cornhusk
cornmeal
cornucopia
corsage
corset
costume
cot
cotton candy
couch
cougar
counter
coverall
cow
coyote
crab
cracker
cranberry
crane
crank
crate
crayfish
crayon
cream
cream cheese
credit card
creme brulee
crepe
crib
croissant
cross
crossbow
crouton
crow
crowbar
crown
crucifix
cruise ship
crumb
crutch
crystal ball
crystal
cuckoo clock
cucumber
cufflink
cummerbund
cup
cupcake
curb
curling iron
curry
curtain
cushion
cutting board
cymbal
dagger
daisy
dalmatian
dandelion
dart
dartboard
dashboard
deer
defibrillator
denture
deodorant
desk
dessert
detonator
dial
diamond
diaper
dice
dip
dirt bike
dish
dishrag
dishwasher
dishwashing liquid
diskette
diving board
dog
dogfood
doghouse
doily
doll
dollhouse
dolly
dolphin
domino
donkey
donut
door
doorbell
doorhandle
doorknob
doorknocker
doormat
doorstop
dough
dragonfly
drain
drawer
dreidel
dress
dresser
drill
drink
drum
drumstick
dryer
duck
duckling
duct
duct tape
duffel bag
dumbbell
dumbwaiter
dumpling
dumpster
duster
dustpan
dynamite
eagle
ear
earplug
earring
earwig
easel
easter egg
eclair
eel
egg
egg roll
eggbeater
eggnog
eggplant
eggshell
elbow
electric chair
elephant
emerald
enchilada
engine
envelope
eraser
espresso
exerciser
exhaust pipe
extinguisher
eye
eye patch
eyedropper
eyeliner
eyepiece
face
face mask
fan
fast food
faucet
feather
fence
fencepost
fern
ferret
ferris wheel
ferry
fig
figurine
file
filing cabinet
film
filter
finger
fingerprint
fire
fire alarm
fire hydrant
fire pit
firecracker
fireplace
firetruck
firewood
fireworks
first-aid kit
fish
fishbowl
fishhook
fishing pole
fishnet stockings
flag
flagpole
flamethrower
flamingo
flan
flashbulb
flashlight
flask
flatiron
flip-flop
flipper
float
floss
flour
flower
flute
fly
flypaper
flyswatter
foam
folder
fondue
food processor
foot
football
football helmet
footbath
footprint
footrest
fork
forklift
fossil
fountain pen
fox
frame
freezer
french fries
french horn
frisbee
frog
fruit
fruitcake
fudge
fungus
funnel
fur coat
furnace
fuse
gallows
game
garbage
garbage truck
gargoyle
garlic
garter
gas mask
gasket
gate
gauge
gauze
gavel
gazelle
gear
gearshift
gel
gem
generator
gift
ginger
gingerbread man
giraffe
girl
glass
glasses
globe
glove
glue
go-kart
goalpost
goat
goblet
goggles
gold
goldfish
golf cart
golf club
gondola
gong
goose
gopher
gorilla
gourd
graffiti
grain
gramophone
granite
granola
grape
grapefruit
grapevine
grass
grasshopper
grate
grater
gravel
gravestone
gravy
green beans
grenade
grill
grille
grinder
grits
groundhog
guacamole
guardrail
guillotine
guinea pig
guitar
gum
gumball
gumdrop
gun
gurney
gutter
gyro
gyroscope
hail
hair
hairbrush
hairdryer
hairnet
hairpin
hairspray
ham
hamburger
hammer
hammock
hamster
hand
handbrake
handcuff
handkerchief
handle
handlebar
handprint
hanger
hard disk
harmonica
harness
harp
hash
hat
hatbox
hatchet
hawk
hay
headband
headdress
headlamp
headlight
headphones
headrest
headscarf
headset
hearing aid
hearse
heater
hedge
hedgehog
helicopter
helmet
highchair
highlighter
hinge
hip
hippopotamus
hobbyhorse
hockey stick
hoe
hole
holster
home plate
honey
honeycomb
honeypot
hood
hoodie
hook
hookah
hopscotch
horn
horse
horseshoe
hose
hot chocolate
hot tub
hot-air balloon
hot-water bottle
hotdog
hotplate
hourglass
hovercraft
hubcap
hula hoop
hummingbird
hummus
humvee
hydrant
hyena
ice
ice cream
ice cube
ice pack
ice-cream cone
icemaker
icepick
iceskate
icicle
iguana
incense
incubator
inhaler
ink
inkwell
insole
iron
ironing board
ivy
jack
jacket
jackhammer
jalapeno
jam
jar
javelin
jeans
jeep
jelly bean
jellyfish
jersey
jet
jetski
jewel
jewelry
jigsaw puzzle
joystick
jug
juice
juicer
jukebox
jump rope
jumpsuit
kale
kaleidoscope
kangaroo
kayak
kazoo
kebab
keg
ketchup
kettle
key
keyboard
keyhole
kilt
kimono
kite
kitten
kiwi
knee
kneepad
knife
knitting
knitting needle
knob
knot
koala
lab coat
ladder
ladle
ladybug
lamb
lamb chop
lamp
lamppost
lampshade
landmine
lantern
lanyard
laptop
lasagna
laser pointer
latch
latte
laundry basket
lava
lavender
lawnmower
leaf
leash
lectern
leech
leek
leg
leggings
lego
lemon
lemonade
lens
leopard
leotard
letter opener
lettuce
license plate
licorice
lid
life jacket
lifesaver
light switch
lightbulb
lighter
lightning bug
lime
limousine
lingerie
lion
lip balm
lip gloss
lipstick
lizard
llama
lobster
lock
locker
locket
log
loincloth
lollipop
loom
loveseat
luggage
lumber
lunchbox
macadamia
macaroni
machete
machine gun
maggot
magnet
magnifier
magnifying glass
mail
mailbox
makeup
mallet
man
manatee
mandolin
mango
manhole
mannequin
mantle
map
maple syrup
marble
margarita
marijuana
marker
marmalade
marshmallow
mascara
mashed potato
mask
mast
mat
match
matchbox
mattress
measuring cup
meat
meat grinder
meatball
meatloaf
medal
meerkat
megaphone
melon
memory stick
metal detector
metronome
microphone
microscope
microwave
milk
milkshake
minivan
mint
mirror
missile
mistletoe
mitten
mixer
moccasin
modem
mold
mole
money
mongoose
monkey
moose
mop
mosquito
mosquito net
moss
moth
motherboard
motorcycle
mouse
mousepad
mousetrap
mousse
mouth
mouthpiece
mud
muff
muffin
mug
mulberry
mulch
mullet
mushroom
music box
mussel
mustache
mustard
nacho
nail
nail clippers
nail file
nail polish
napkin
napkin ring
navel
neck
necklace
needle
nest
net
nightshirt
nightstand
noisemaker
noodle
noose
nose
notebook
notepad
nut
nutcracker
oar
oatmeal
octopus
odometer
oil
oilcan
okra
olive
omelet
onion
orange
orange rind
orangutan
orchid
organ
origami
ostrich
otter
ottoman
outfit
outlet
oven
overalls
owl
oyster
pacifier
paddle
padlock
paint
paintbrush
painting
pajamas
palette
pallet
palm tree
pan
pancake
panda
panther
panties
pants
pantsuit
pantyhose
papaya
paper
paper bag
paper plate
paper towel
paperclip
paperweight
parachute
parfait
parking meter
parrot
parsley
pasta
pastry
patch
patty
payphone
pea
peach
peacock
peanut
peanut butter
pear
pearl
pecan
pedal
pedometer
peeler
peg
pelican
pen
pencil
pencil sharpener
pendulum
penguin
penholder
penlight
pennant
pepper mill
pepper
peppermint
pepperoni
perfume
periscope
pesto
pet food
petal
petri dish
pheasant
phone
phone booth
photo booth
photograph
piano
pickax
pickle
pie
piecrust
pig
pigeon
piggy bank
piglet
pill
pillbox
pillow
pin
pinball
pincushion
pine needle
pine tree
pineapple
pinecone
ping-pong table
pinwheel
pipe
pistachio
pita
pitcher
pitchfork
pizza
place mat
plant
plaster cast
plastic film
plate
platypus
playing card
playpen
pliers
plug
plum
plunger
pocket
pocket watch
pocketknife
pogo stick
poinsettia
poker
polar bear
polaroid
pole
police car
polisher
polo shirt
polygraph
pom-pom
pomegranate
pony
poodle
pool table
popcorn
poppy
popsicle
porcupine
porthole
possum
poster
pot
potato
potholder
pothole
potpie
potpourri
powder
power line
praying mantis
pretzel
printer
prism
projector
propeller
prune
puck
pudding
puddle
puffin
pug
pulley
pulpit
pump
pumpkin
punch
punching bag
puppet
puppy
purse
quad
quesadilla
quiche
quill
quilt
rabbit
raccoon
racehorse
rack
racket
radar
radiator
radio
radish
raft
rag
railing
rain gauge
raincoat
raisin
rake
ram
ramp
raspberry
rat
ratchet
rattle
rattlesnake
ravioli
razor
razor blade
ready meal
rearview mirror
recliner
record
record player
recorder
red carpet
reel
refrigerator
reindeer
remote control
retainer
revolver
revolving door
rhinoceros
rhubarb
ribbon
rice
rickshaw
rifle
rim
ring
riser
road sign
roadsweeper
robe
robot
rock
rocket
rocking chair
rocking horse
roll
roller
roller coaster
rollerblade
rollerskate
rolling pin
roof rack
rooster
root
rope
rosary
rose
roulette wheel
router
rubber band
ruby
rudder
rug
ruler
rust
saddle
safety pin
saffron
sail
sailboat
salad
salami
saltshaker
sand
sandal
sandbag
sandbox
sandcastle
sandpaper
sandwich
sarcophagus
sardine
satellite
satellite dish
sauce
saucer
sauerkraut
sausage
saw
sawhorse
saxophone
scaffold
scaffolding
scale
scallion
scallop
scalpel
scanner
scarecrow
scarf
school bus
scissors
scone
scoop
scooter
scoreboard
scorpion
scrabble
scrambled egg
scraper
screen
screw
screwdriver
scuba
sea urchin
seafood
seagull
seahorse
seal
seaplane
seatbelt
seaweed
seed
seesaw
seismograph
sequin
sewage
sewing kit
sewing machine
shaker
shark
shaving cream
shawl
shears
sheath
sheep
sheet
shelf
shell
shield
ship
shirt
shoe
shoe polish
shoehorn
shoelace
shopping basket
shopping cart
shortbread
shorts
shoulder
shovel
shower
shower cap
shower curtain
showerhead
shredder
shrimp
shuffleboard
shutter
sickle
sidecar
sifter
silicone
silverware
sim card
sink
siren
skateboard
skeleton
skewer
ski
ski boots
ski lift
ski pole
skin
skirt
skull
skunk
sled
sledgehammer
sleeping bag
slicer
slide
slime
sling
slingshot
slipper
slot
slot machine
sloth
slug
smoke alarm
smoothie
snack
snail
snake
snorkel
snow
snowball
snowboard
snowman
snowmobile
snowplow
snowshoe
snowsuit
soap
soccer ball
sock
soda
soda fountain
sofa bed
solar panel
soldering iron
sombrero
sonogram
sorbet
souffle
soup
soy sauce
space shuttle
spacesuit
spaghetti
spam
spareribs
spark plug
sparkler
spatula
speaker
spear
speedboat
speedometer
spider
spider web
spinach
splinter
sponge
spool
spoon
spout
spring roll
springboard
sprinkler
sprouts
spur
squash
squeegee
squid
squirrel
squirt gun
stained glass
stair
stake
stalagmite
stamp
staple
staple gun
stapler
star fruit
starfish
station wagon
statue
steak
steamroller
steering wheel
stem
step stool
stereo
stethoscope
stew
stick
sticker
stiletto
stilt
stingray
stir fry
stirrup
stockings
stomach
stool
stopwatch
stove
straightjacket
strainer
strap
straw
strawberry
streetlight
stretcher
string cheese
stroller
stuffing
stump
submarine
subway
sugar cube
suit
suitcase
sundae
sundial
sunflower
sunglasses
sunroof
surfboard
sushi
suspenders
swab
swan
sweater
sweatsuit
sweeper
sweet potato
swimming pool
swimsuit
swing
swing set
switch
swizzle stick
sword
swordfish
syringe
syrup
t-shirt
tab
table
tablecloth
tablet
tack
tackle
taco
tadpole
taffy
tag
taillight
tamale
tambourine
tank
tape
tape measure
tape recorder
tapestry
tarantula
target
tarp
tattoo
taxi
tea
teabag
teacup
teapot
teddy bear
tee
teepee
telegraph
telephone pole
telescope
television
tennis ball
tent
terrarium
test tube
thermometer
thermos
thermostat
thimble
thorn
thread
throne
thumb
thumbtack
tiara
tick
ticktacktoe
tie
tiger
tile
timer
tinsel
tiramisu
toad
toast
toaster
toaster oven
toe
toga
toilet
toilet paper
tomato
tomato sauce
tongs
tongue
tool
toolbox
tooth
toothbrush
toothpaste
toothpick
top hat
torch
torpedo
torso
tortellini
tortilla
tostada
totem pole
toucan
touchpad
towel
towel rack
toy
tractor
traffic light
trailer
train
train car
train set
trampoline
trap
trapdoor
trashcan
tray
treadmill
treasure
tree
tree trunk
triangle
tricycle
trident
trigger
tripod
trolley
trombone
trophy
trough
trowel
truck
trumpet
trunk
tuba
tube top
tugboat
tulip
tumbleweed
tuning fork
tupperware
turban
turbine
turf
turkey
turnstile
turntable
turtle
turtleneck
tuxedo
tweezers
twig
typewriter
ukulele
umbrella
undershirt
underwear
unicycle
uniform
urinal
urn
vacuum
valve
van
vase
vegetable
veil
velcro
vending machine
vent
vest
vial
videocassette
videogame
viewfinder
violin
vise
visor
volleyball
vulture
wafer
waffle
waffle iron
wagon
walker
wall
wallet
wallpaper
walnut
walrus
wand
warthog
washboard
washcloth
washing machine
wasp
watch
water bottle
water cooler
water filter
water fountain
water heater
watering can
watermelon
waterwheel
wax
wax paper
weasel
weather vane
webcam
wedding cake
wedge
weed
wetsuit
whale
wheat
wheel
wheelbarrow
wheelchair
whip
whipped cream
whisk
whistle
whiteboard
whoopee cushion
wick
wig
wind chimes
window
windowsill
windshield
windshield wiper
windsock
wine
wine bottle
wine cooler
wineglass
wing
wire
wire cutters
wok
wolf
woman
wood
wooden leg
workbench
worm
wrap
wrapping paper
wreath
wreck
wrench
wrist
xylophone
yacht
yak
yarn
yo-yo
yogurt
yoke
yolk
zebra
zipper
zucchini
water
bicycle
sun
moon
star
rain
mountain
river
ocean
nothing
everything
the internet
wifi
air
love
time
music
silence
sleep
a dream
electricity
a hug
the weather
the future
the past
math
zero
infinity
luck
a secret
happiness
boredom
friendship
a joke
a song
a language
the alphabet
a shadow
a rainbow
an echo
a smell
a color
gravity
an idea
the news
a game
a memory

In [ ]:
%%writefile wildcards.txt
nothing
everything
the internet
wifi
air
love
time
music
silence
sleep
a dream
electricity
money
a hug
the weather
the future
the past
math
zero
infinity
luck
a secret
happiness
boredom
friendship
a joke
a song
a language
the alphabet
a shadow
a rainbow
an echo
a smell
a color
gravity
an idea
the news
a game
a memory

In [ ]:
%%writefile guesser.py
"""
20Q-style word guesser for an Arduino Uno + I2C LCD.

The laptop does the thinking, the Uno just shows text.

How it works (same ideas as the 2003 Radica 20Q toy):
  * 5 answer levels: yes / probably / sometimes / probably not / no
  * every word keeps a SCORE instead of being deleted on a bad answer,
    so one sloppy answer doesn't ruin the game
  * it asks the question that best splits the likely words
  * it keeps asking and guessing again after a wrong guess
  * it LEARNS: new words and corrected answers are saved to words.json

Run:     python guesser.py
Grow it: python guesser.py prompt newwords.txt   (writes fill_prompt.txt)
         python guesser.py merge fill_result.json
"""
import json
import math
import os
import sys
import time

try:
    import serial
except ImportError:
    serial = None

# ---------------------------------------------------------------- settings
PORT = "COM13"
BAUD = 9600
HERE = os.path.dirname(os.path.abspath(__file__))
DATA_FILE = os.path.join(HERE, "words.json")

MIN_Q = 5            # earliest a confident guess is allowed
CONFIDENT = 0.60     # guess early when the top word has this probability
GUESS_AT = (20, 23, 26, 29)   # forced guesses, like 20Q
MAX_Q = 30
CANDIDATES = 250     # only the likeliest words are used to choose questions
SEED_CONF = 3        # how strongly the built-in answers resist being changed

LEVELS = (1.0, 0.75, 0.5, 0.25, 0.0)
ANSWERS = {"1": 1.0, "y": 1.0, "2": 0.75, "p": 0.75, "3": 0.5, "s": 0.5,
           "d": 0.5, "4": 0.25, "m": 0.25, "5": 0.0, "n": 0.0}
LEGEND = "  1=yes  2=probably  3=sometimes/unsure  4=probably not  5=no  (q=quit)"

# ----------------------------------------------------------- built-in data
# Questions must fit on a 16x2 LCD (32 characters max).
QUESTIONS = {
    "alive": "Is it alive?",
    "animal": "Is it an animal?",
    "plant": "Is it a plant?",
    "manmade": "Is it man-made?",
    "hand": "Fits in your hand?",
    "bigcar": "Bigger than a car?",
    "bigperson": "Bigger than a person?",
    "electric": "Does it use electricity?",
    "edible": "Can you eat it?",
    "sweet": "Does it taste sweet?",
    "home": "Found in most homes?",
    "kitchen": "Found in a kitchen?",
    "outdoors": "Found outdoors?",
    "wheels": "Does it have wheels?",
    "noise": "Does it make noise?",
    "moves": "Can it move by itself?",
    "flies": "Can it fly?",
    "water": "Is it found in water?",
    "metal": "Is it made of metal?",
    "wood": "Is it made of wood?",
    "fabric": "Is it made of fabric?",
    "glass": "Is it made of glass?",
    "plastic": "Is it made of plastic?",
    "wear": "Do you wear it?",
    "tool": "Is it a tool?",
    "toy": "Is it a toy or game?",
    "pet": "Is it a common pet?",
    "fur": "Does it have fur or hair?",
    "legs": "Does it have legs?",
    "hot": "Can it get hot?",
    "cold": "Is it often cold?",
    "sit": "Can you sit on it?",
    "read": "Do you read it?",
    "screen": "Does it have a screen?",
    "sharp": "Is it sharp?",
    "round": "Is it round?",
    "school": "Used at school?",
    "sport": "Used in sports?",
    "music": "Used for music?",
    "danger": "Can it hurt you?",
    "sky": "Is it in the sky?",
    "liquid": "Is it a liquid?",
    "soft": "Is it soft?",
    "vehicle": "Is it a vehicle?",
}

# word -> tags that are YES.  "~tag" = sometimes.  Anything not listed = no.
SEED = {
    "dog": "alive animal fur legs moves noise pet outdoors home soft ~danger",
    "cat": "alive animal fur legs moves noise pet home outdoors soft",
    "horse": "alive animal fur legs moves noise outdoors bigperson ~danger ~sport",
    "cow": "alive animal fur legs moves noise outdoors bigperson",
    "elephant": "alive animal legs moves noise outdoors bigperson ~bigcar danger",
    "fish": "alive animal water moves ~edible ~pet ~hand",
    "bird": "alive animal flies legs moves noise outdoors sky hand ~pet",
    "snake": "alive animal moves outdoors danger ~pet ~water",
    "shark": "alive animal water moves danger bigperson",
    "spider": "alive animal legs moves outdoors hand ~danger",
    "frog": "alive animal legs moves water outdoors hand ~noise ~pet",
    "tree": "alive plant outdoors wood bigperson ~edible",
    "flower": "alive plant outdoors soft hand ~home",
    "grass": "alive plant outdoors soft",
    "apple": "~plant edible sweet hand round kitchen home",
    "banana": "~plant edible sweet hand kitchen home soft",
    "pizza": "manmade edible kitchen home hot ~round",
    "bread": "manmade edible kitchen home soft hand ~sweet",
    "cheese": "manmade edible kitchen home soft hand",
    "egg": "edible kitchen home hand round ~hot",
    "milk": "manmade edible liquid kitchen home ~cold ~sweet",
    "water": "~edible liquid home kitchen outdoors water ~cold ~hot",
    "ice cream": "manmade edible sweet cold kitchen home soft ~hand",
    "chocolate": "manmade edible sweet hand home kitchen ~soft",
    "cookie": "manmade edible sweet hand kitchen home round",
    "pencil": "manmade hand home school wood sharp",
    "pen": "manmade hand home school plastic ~metal",
    "book": "manmade hand home school read",
    "paper": "manmade hand home school read",
    "scissors": "manmade hand home school metal tool sharp ~plastic",
    "knife": "manmade hand kitchen home metal tool sharp danger",
    "spoon": "manmade hand kitchen home metal tool ~plastic",
    "fork": "manmade hand kitchen home metal tool sharp ~plastic",
    "cup": "manmade hand kitchen home ~glass ~plastic",
    "plate": "manmade kitchen home round ~hand ~glass",
    "bottle": "manmade hand kitchen home ~glass ~plastic",
    "key": "manmade hand home metal ~tool",
    "phone": "manmade hand home electric screen noise glass plastic metal",
    "laptop": "manmade electric screen home school plastic metal hot ~noise",
    "computer": "manmade electric screen home school plastic metal hot ~noise",
    "television": "manmade electric screen home noise glass plastic hot",
    "clock": "manmade home noise round ~electric",
    "lamp": "manmade home electric glass metal hot ~plastic",
    "fan": "manmade home electric noise plastic metal ~moves",
    "refrigerator": "manmade home kitchen electric noise cold metal ~bigperson",
    "chair": "manmade home sit wood school ~metal ~plastic ~fabric",
    "couch": "manmade home sit soft fabric ~bigperson ~wood",
    "bed": "manmade home sit soft fabric ~bigperson ~wood ~metal",
    "table": "manmade home kitchen school wood ~metal ~glass",
    "door": "manmade home wood bigperson ~metal ~glass",
    "window": "manmade home glass",
    "mirror": "manmade home glass ~hand",
    "shoe": "manmade wear home fabric ~outdoors ~hand ~soft",
    "shirt": "manmade wear home fabric soft",
    "hat": "manmade wear home fabric soft ~outdoors",
    "glasses": "manmade wear hand glass plastic home",
    "watch": "manmade wear hand metal electric ~screen ~noise round ~glass",
    "backpack": "manmade ~wear school fabric home ~outdoors soft",
    "umbrella": "manmade ~hand outdoors fabric metal home ~tool",
    "ball": "manmade toy sport round ~hand outdoors ~plastic",
    "bicycle": "manmade wheels vehicle outdoors metal sport",
    "car": "manmade wheels vehicle outdoors metal moves electric noise bigperson",
    "bus": "manmade wheels vehicle outdoors metal moves noise bigperson bigcar",
    "train": "manmade wheels vehicle outdoors metal moves noise bigperson bigcar",
    "airplane": "manmade vehicle flies sky metal moves noise bigperson bigcar",
    "rocket": "manmade vehicle flies sky metal moves noise bigperson bigcar hot danger",
    "helicopter": "manmade vehicle flies sky metal moves noise bigperson ~bigcar",
    "boat": "manmade vehicle water moves ~noise bigperson ~bigcar ~wood ~metal",
    "guitar": "manmade music wood noise home",
    "piano": "manmade music wood home noise bigperson ~metal",
    "drum": "manmade music noise round ~wood",
    "hammer": "manmade hand tool metal wood home ~danger",
    "screwdriver": "manmade hand tool metal home ~sharp",
    "saw": "manmade hand tool metal sharp danger ~wood",
    "battery": "manmade hand electric metal ~home",
    "light bulb": "manmade hand electric glass home hot round",
    "robot": "manmade electric moves metal plastic ~noise ~toy ~screen ~legs ~wheels",
    "sun": "sky hot round outdoors bigcar bigperson danger",
    "moon": "sky round outdoors bigcar bigperson cold",
    "star": "sky hot round outdoors bigcar bigperson",
    "cloud": "sky outdoors ~water soft bigcar bigperson ~cold",
    "rain": "water liquid outdoors sky ~cold ~noise",
    "snow": "water outdoors cold ~sky soft",
    "rock": "outdoors ~hand ~bigperson",
    "mountain": "outdoors bigcar bigperson",
    "river": "water liquid outdoors bigcar bigperson ~noise ~moves",
    "ocean": "water liquid outdoors bigcar bigperson ~noise ~moves",
    "fire": "hot ~outdoors danger ~noise ~moves ~home",
    "sand": "outdoors ~water ~soft ~hand",
    "toothbrush": "manmade hand home plastic ~soft",
    "soap": "manmade hand home ~liquid",
    "towel": "manmade home fabric soft",
    "pillow": "manmade home soft fabric",
    "blanket": "manmade home soft fabric ~wear",
    "coin": "manmade hand metal round ~home",
    "candle": "manmade hand home hot ~danger",
    "camera": "manmade hand electric screen glass plastic ~noise home",
    "headphones": "manmade electric wear music noise plastic home ~hand",
}


def build_seed():
    words = {}
    for name, tags in SEED.items():
        vals = {q: 0.0 for q in QUESTIONS}
        for tag in tags.split():
            if tag.startswith("~"):
                vals[tag[1:]] = 0.5
            else:
                vals[tag] = 1.0
        words[name] = {"v": vals, "n": {q: SEED_CONF for q in QUESTIONS},
                       "plays": 1}
    return {"questions": dict(QUESTIONS), "words": words, "games": 0, "wins": 0}


# ------------------------------------------------------------- save / load
def load():
    if os.path.exists(DATA_FILE):
        with open(DATA_FILE, encoding="utf-8") as f:
            return json.load(f)
    return build_seed()


def save(db):
    tmp = DATA_FILE + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(db, f)
    os.replace(tmp, DATA_FILE)


# ----------------------------------------------------------------- display
def split_lines(text):
    if len(text) <= 16:
        return text, ""
    cut = text.rfind(" ", 0, 17)
    cut = cut if cut > 0 else 16
    return text[:cut][:16], text[cut:].strip()[:16]


class Display:
    """Sends text to the Uno. If the Uno isn't there, it still prints."""

    def __init__(self):
        self.ser = None
        if serial is None:
            print("(pyserial not installed - running without the LCD)")
            return
        try:
            self.ser = serial.Serial(PORT, BAUD, timeout=1)
            time.sleep(2)  # the Uno resets when the port opens
        except Exception as err:
            print(f"(could not open {PORT}: {err})")
            print("(running without the LCD - close the Serial Monitor?)")

    def show(self, text):
        top, bottom = split_lines(text)
        print(f"  [LCD] {top} / {bottom}")
        if self.ser:
            self.ser.write(f"{top}|{bottom}\n".encode())

    def close(self):
        if self.ser:
            self.ser.close()


# ------------------------------------------------------------------- input
def ask_answer(text):
    """Return an answer level 0..1 for a question."""
    while True:
        a = input(f"{text}\n{LEGEND}\n> ").strip().lower()
        if a == "q":
            raise KeyboardInterrupt
        if a in ANSWERS:
            return ANSWERS[a]


def yes_no(text):
    while True:
        a = input(text).strip().lower()
        if a == "q":
            raise KeyboardInterrupt
        if a in ("y", "yes", "1"):
            return True
        if a in ("n", "no", "5"):
            return False


def read_line(text):
    return input(text).strip()


# ----------------------------------------------------------------- the AI
def val(word, q):
    return word["v"].get(q, 0.5)      # unknown = 0.5


def lik(a, v):
    """How well does answer a fit a word whose stored value is v?"""
    return 0.05 + 0.95 * (1 - abs(a - v))


def posterior(logp):
    m = max(logp.values())
    e = {w: math.exp(s - m) for w, s in logp.items()}
    z = sum(e.values())
    return {w: x / z for w, x in e.items()}


def entropy(ps):
    return -sum(p * math.log(p) for p in ps if p > 0)


def pick_question(db, post, asked):
    """Choose the question that is expected to narrow things down the most."""
    ranked = sorted(post.items(), key=lambda kv: -kv[1])[:CANDIDATES]
    z = sum(p for _, p in ranked)
    cands = [(db["words"][w], p / z) for w, p in ranked]
    best_q = None
    best_h = entropy([p for _, p in cands]) - 0.01   # must help at least a bit
    for q in db["questions"]:
        if q in asked:
            continue
        rows = []
        tot = [0.0] * 5
        for word, p in cands:
            v = val(word, q)
            ls = [lik(a, v) for a in LEVELS]
            s = sum(ls)
            row = [p * x / s for x in ls]
            rows.append(row)
            for i, x in enumerate(row):
                tot[i] += x
        h = 0.0
        for i in range(5):
            if tot[i] > 0:
                h += tot[i] * entropy([r[i] / tot[i] for r in rows])
        if h < best_h:
            best_q, best_h = q, h
    return best_q


def play(db, lcd):
    """One game. Returns (guessed_word_or_None, answers_given, won)."""
    words = db["words"]
    logp = {w: math.log(1 + d["plays"]) for w, d in words.items()}
    asked = {}
    rejected = set()
    last_guess_n = -1

    lcd.show("Think of a word!")
    read_line("Press Enter when ready...")

    while True:
        n = len(asked)
        post = posterior(logp)
        top, p_top = max(post.items(), key=lambda kv: kv[1])
        want_guess = n != last_guess_n and (
            (n >= MIN_Q and p_top >= CONFIDENT) or n in GUESS_AT)

        q = None
        if not want_guess and n < MAX_Q:
            q = pick_question(db, post, asked)
        if q is not None:
            text = db["questions"][q]
            lcd.show(text)
            a = ask_answer(f"Q{n + 1}: {text}")
            asked[q] = a
            for w, d in words.items():
                logp[w] += math.log(lik(a, val(d, q)))
            continue

        # time to guess, or nothing useful left to ask
        if n == last_guess_n or top in rejected:
            lcd.show("You got me!")
            return None, asked, False
        last_guess_n = n
        lcd.show(f"Is it {top}?")
        if yes_no(f"Is it {top}? (y/n) > "):
            lcd.show("I win!")
            return top, asked, True
        rejected.add(top)
        logp[top] -= 8


def learn(db, name, asked):
    """Fold a finished game into what the word 'looks like'."""
    w = db["words"].setdefault(name, {"v": {}, "n": {}, "plays": 0})
    w["plays"] += 1
    for q, a in asked.items():
        n = w["n"].get(q, 0)
        old = w["v"].get(q, a)
        w["v"][q] = old + (a - old) / (n + 1)
        w["n"][q] = n + 1


# --------------------------------------------- growing the word list (LLM)
def make_prompt(path):
    db = load()
    with open(path, encoding="utf-8") as f:
        new = [line.strip().lower() for line in f if line.strip()]
    new = [w for w in dict.fromkeys(new) if w not in db["words"]]
    qs = list(db["questions"].values())
    lines = [
        "For each word below, answer every question about the thing the word",
        "names. Use one digit per question: 4=yes, 3=probably, 2=sometimes or",
        "depends, 1=probably not, 0=no.",
        f"Reply with ONLY a JSON object mapping each word to a string of exactly",
        f"{len(qs)} digits, one per question, in the numbered order. No other text.",
        "", "Questions:",
    ]
    lines += [f"{i + 1}. {t}" for i, t in enumerate(qs)]
    lines += ["", "Words:"] + new
    out = os.path.join(HERE, "fill_prompt.txt")
    with open(out, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    print(f"Wrote {out} with {len(new)} new words.")
    print("Paste it into a Claude chat (30-50 words per paste works best),")
    print("save the JSON reply as fill_result.json, then run:")
    print("  python guesser.py merge fill_result.json")


def merge(path):
    db = load()
    with open(path, encoding="utf-8") as f:
        text = f.read().strip()
    if text.startswith("```"):
        text = text.split("\n", 1)[1].rsplit("```", 1)[0]
    data = json.loads(text)
    qids = list(db["questions"])
    added = 0
    for word, digits in data.items():
        digits = str(digits).strip()
        if len(digits) != len(qids) or any(c not in "01234" for c in digits):
            print(f"skipped '{word}': need exactly {len(qids)} digits 0-4")
            continue
        db["words"][word.strip().lower()] = {
            "v": {q: int(c) / 4 for q, c in zip(qids, digits)},
            "n": {q: 2 for q in qids},
            "plays": 1,
        }
        added += 1
    save(db)
    print(f"Added {added} words. The list now has {len(db['words'])}.")


# -------------------------------------------------------------------- main
def main():
    args = sys.argv[1:]
    if len(args) == 2 and args[0] == "prompt":
        return make_prompt(args[1])
    if len(args) == 2 and args[0] == "merge":
        return merge(args[1])

    db = load()
    lcd = Display()
    print(f"I know {len(db['words'])} words. Think of an object, animal or food.")
    try:
        while True:
            guess, asked, won = play(db, lcd)
            db["games"] += 1
            if won:
                db["wins"] += 1
                learn(db, guess, asked)
            else:
                name = read_line("What were you thinking of? > ").lower()
                if name:
                    learn(db, name, asked)
                    lcd.show("Learned it!")
            save(db)
            if not yes_no("Play again? (y/n) > "):
                break
    except (KeyboardInterrupt, EOFError):
        print()
    finally:
        save(db)
        if db["games"]:
            print(f"Record: {db['wins']} wins in {db['games']} games.")
        lcd.close()


if __name__ == "__main__":
    main()


In [ ]:
%%writefile score_words.py
"""
Build the word table: score every word in word_list.txt on every question
with a local model, using every GPU in the session.

word_list.txt is the THINGS concept list (Hebart et al., 2019, PLoS ONE)
plus a few extra words.

Settings come from environment variables (set in the notebook with %env):
    LIMIT        words to score this run; 0 = all.  Default 40 (a test run)
    MODEL        Ollama model.                       Default gemma4:12b

Progress is saved to scores.jsonl after every word, so a restart picks up
where it stopped. The finished table is written to words.json.
"""
import json
import os
import random
import re
import subprocess
import threading
import time
from concurrent.futures import ThreadPoolExecutor

import ollama

from questions import QUESTIONS

MODEL = os.environ.get("MODEL", "gemma4:12b")
LIMIT = int(os.environ.get("LIMIT", "40"))
PARALLEL = 2          # requests each GPU works on at once
OUT = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."

LABELS = ["yes", "probably", "sometimes", "probably not", "no"]
VALUE = {"yes": 1.0, "probably": 0.75, "sometimes": 0.5,
         "probably not": 0.25, "no": 0.0}

SCORE_SCHEMA = {
    "type": "object",
    "properties": {q: {"type": "string", "enum": LABELS} for q in QUESTIONS},
    "required": list(QUESTIONS),
}

# Fixed text first and the word last, so the repeated part can be reused.
HEADER = (
    "You are helping build a 20 Questions game. For the thing named by the "
    "word at the end (it may be an object, animal, food, place, idea or "
    "activity), answer every question about a TYPICAL example of it "
    "using exactly one of: yes, probably, sometimes, probably not, no.\n"
    "Use 'yes' only if it is true for almost every example, 'no' only if it "
    "is false for almost every example, and 'sometimes' when it depends or "
    "is about half and half.\n\nQuestions:\n"
    + "\n".join(f"{q}: {t}" for q, t in QUESTIONS.items())
    + "\n\nWord: "
)


# ----------------------------------------------------------- GPUs, servers
def count_gpus():
    if os.environ.get("GPUS"):
        return int(os.environ["GPUS"])
    try:
        out = subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                             text=True).stdout
    except FileNotFoundError:
        return 1
    return max(1, sum(1 for line in out.splitlines() if line.startswith("GPU ")))


def make_client(port):
    return ollama.Client(host=f"http://127.0.0.1:{port}")


def server_up(port):
    try:
        make_client(port).list()
        return True
    except Exception:
        return False


def start_servers(n_gpus):
    """One Ollama server per GPU, each pinned to its own card."""
    ports = [11434 + g for g in range(n_gpus)]
    for g, port in enumerate(ports):
        if not server_up(port):
            env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(g),
                       OLLAMA_HOST=f"127.0.0.1:{port}",
                       OLLAMA_KEEP_ALIVE="-1",
                       OLLAMA_NUM_PARALLEL=str(PARALLEL))
            log = open(os.path.join(OUT, f"ollama_{port}.log"), "w")
            subprocess.Popen(["ollama", "serve"], env=env, stdout=log,
                             stderr=subprocess.STDOUT)
    for port in ports:
        for _ in range(60):
            if server_up(port):
                break
            time.sleep(1)
        else:
            raise RuntimeError(f"Ollama server on port {port} did not start")
    return ports


def pull_model(port):
    env = dict(os.environ, OLLAMA_HOST=f"127.0.0.1:{port}")
    try:
        subprocess.run(["ollama", "pull", MODEL], env=env, check=True)
    except FileNotFoundError:
        print("(ollama command not found, skipping the model download)")


# --------------------------------------------------------------- word list
def load_word_list():
    path = "word_list.txt"
    with open(path, encoding="utf-8") as f:
        return [w.strip() for w in f.read().split("\n") if w.strip()]


# ----------------------------------------------------------------- scoring
def score_word(client, word):
    last = None
    for _ in range(3):
        try:
            reply = client.chat(
                model=MODEL,
                messages=[{"role": "user", "content": HEADER + word}],
                format=SCORE_SCHEMA, think=False,
                options={"temperature": 0, "num_ctx": 4096})
            raw = json.loads(reply["message"]["content"])
            return {q: VALUE[raw[q]] for q in QUESTIONS}
        except Exception as err:
            last = err
            time.sleep(2)
    raise RuntimeError(f"{word}: {last}")


def load_scores():
    scores = {}
    path = os.path.join(OUT, "scores.jsonl")
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    row = json.loads(line)
                    if set(row["v"]) == set(QUESTIONS):   # skip old-question rows
                        scores[row["word"]] = row["v"]
    return scores


def load_wildcards():
    try:
        with open("wildcards.txt", encoding="utf-8") as f:
            return {w.strip() for w in f.read().split("\n") if w.strip()}
    except FileNotFoundError:
        return set()


WILDCARDS = load_wildcards()


def popularity(word):
    """Wildcards (nothing, the internet...) stay rare surprises. Common words get a bigger starting weight, so the game guesses 'dog'
    before 'dingo'. Uses the wordfreq package if it is installed."""
    if word in WILDCARDS:
        return 3
    try:
        from wordfreq import zipf_frequency
    except ImportError:
        return 1
    zipf = zipf_frequency(word, "en")
    return 1 + int(min(300, 10 ** max(0.0, zipf - 3)))


def write_db(scores):
    db = {"questions": dict(QUESTIONS), "words": {}, "games": 0, "wins": 0}
    for word, v in scores.items():
        db["words"][word] = {"v": v, "n": {q: 2 for q in QUESTIONS},
                             "plays": popularity(word)}
    path = os.path.join(OUT, "words.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(db, f)
    print(f"Wrote {path} with {len(scores)} words.")


def main():
    n_gpus = count_gpus()
    print(f"GPUs: {n_gpus}   model: {MODEL}")
    ports = start_servers(n_gpus)
    clients = [make_client(p) for p in ports]
    pull_model(ports[0])

    words = load_word_list()
    if LIMIT:
        words = random.Random(1).sample(words, min(LIMIT, len(words)))

    done = load_scores()
    todo = [w for w in words if w not in done]
    print(f"{len(words)} words wanted, {len(done)} already scored, "
          f"{len(todo)} to do.")

    lock = threading.Lock()
    path = os.path.join(OUT, "scores.jsonl")
    start = time.time()
    finished = 0
    failed = []

    def job(item):
        nonlocal finished
        i, word = item
        try:
            v = score_word(clients[i % len(clients)], word)
        except Exception as err:
            with lock:
                failed.append(word)
                print(f"  FAILED {err}")
            return
        with lock:
            with open(path, "a", encoding="utf-8") as f:
                f.write(json.dumps({"word": word, "v": v}) + "\n")
            done[word] = v
            finished += 1
            if finished % 10 == 0 or finished == len(todo):
                secs = (time.time() - start) / finished
                left = secs * (len(todo) - finished) / 3600
                print(f"  {finished}/{len(todo)} done, {secs:.1f} s/word, "
                      f"about {left:.1f} hours left", flush=True)

    try:
        with ThreadPoolExecutor(len(clients) * PARALLEL) as ex:
            list(ex.map(job, enumerate(todo)))
    finally:
        write_db(load_scores())
    if failed:
        print(f"{len(failed)} words failed; run again to retry them.")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile analyze_table.py
"""Check the finished table: weak questions, near-duplicate words, and a
simulated game. Run after score_words.py."""
import contextlib
import io
import json
import os
import random

import numpy as np

import guesser

OUT = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
N_SIM = int(os.environ.get("N_SIM", "60"))

with open(os.path.join(OUT, "words.json"), encoding="utf-8") as f:
    db = json.load(f)
names = list(db["words"])
qids = list(db["questions"])
M = np.array([[db["words"][w]["v"][q] for q in qids] for w in names])
print(f"{len(names)} words x {len(qids)} questions")
if len(names) < 150:
    print("(only a small sample so far, so the checks below are rough)")

# 1. questions nearly everything answers the same way help little
yes = (M >= 0.75).mean(axis=0)
order = np.argsort(yes)
print("\nQuestions that are almost always 'no' or almost always 'yes':")
if len(names) < 300:
    print("  (skipped: with so few words, rarely-true questions look useless "
          "even when they aren't. Run the full table to judge them.)")
else:
    weak = [(db["questions"][qids[i]], yes[i]) for i in order
            if yes[i] < 0.01 or yes[i] > 0.85]
    for text, y in weak[:20]:
        print(f"  {y:4.1%} yes   {text}")
    if not weak:
        print("  none")

# 2. words with almost identical answers can't be told apart
close = []
for i in range(0, len(names), 40):
    d = np.abs(M[i:i + 40, None, :] - M[None, :, :]).mean(axis=2)
    for r in range(d.shape[0]):
        d[r, i + r] = 9
    j = d.argmin(axis=1)
    for r in range(d.shape[0]):
        if d[r, j[r]] < 0.04:
            close.append((names[i + r], names[j[r]]))
print(f"\nWords with a near-identical twin: {len(close)} of {len(names)}")
for a, b in close[:12]:
    print(f"  {a}  ~  {b}")

# 3. simulated games (answers come from the table itself, with noise)
rev = {t: q for q, t in db["questions"].items()}


class Lcd:
    def show(self, text):
        pass


def simulate(noise, games):
    rng = random.Random(5)
    wins = asked_total = 0
    for _ in range(games):
        target = rng.choice(names)
        word = db["words"][target]

        def ask(text):
            q = rev[text.split(": ", 1)[1]]
            v = word["v"][q]
            if rng.random() < noise:
                v = min(1, max(0, v + rng.choice([-.5, -.25, .25, .5])))
            return v

        guesser.ask_answer = ask
        guesser.yes_no = lambda text: text.split("Is it ")[1].split("?")[0] == target
        guesser.read_line = lambda text: ""
        with contextlib.redirect_stdout(io.StringIO()):
            _, asked, won = guesser.play(db, Lcd())
        wins += won
        asked_total += len(asked)
    print(f"  noise {noise:.0%}: won {wins}/{games}, "
          f"{asked_total / games:.1f} questions on average")


print(f"\nSimulated games ({N_SIM} each):")
for noise in (0.0, 0.1, 0.2):
    simulate(noise, N_SIM)
print("\nThese games use the table's own answers, so real people will do "
      "worse. The real test is playing it.")


In [ ]:
# Remove results from any earlier version of this notebook (old questions)
!rm -f scores.jsonl words.json words.txt /kaggle/working/scores.jsonl /kaggle/working/words.json /kaggle/working/words.txt


In [ ]:
# TEST RUN. Change LIMIT=40 to LIMIT=0 for the full run (see the top of this notebook).
# Don't rerun the cell above between the test run and the full run, or the test results are deleted.
%env LIMIT=40
!python score_words.py


In [ ]:
!python analyze_table.py